# One Look · 05 · Piste VLM (Colab, GPU)

**Hypothèse du mentor (29 oct) :** un petit modèle vision-langage, qui « connaît » le monde, distinguerait mieux l'eau habituelle (mer, fleuves, rizières) d'une inondation.

Le VLM voit la tuile deux fois (couleurs naturelles, infrarouge court) et choisit une lettre : A normal, B inondation, C incendie. On lit ses scores pour les trois lettres en une seule passe ; ils deviennent des probabilités, traitées comme celles du réseau (seuil « inconnu » calibré sur la validation, mêmes mesures). Code : `src/onelook/models/vlm.py`, `scripts/vlm.py`.

| Run | Modèle | Ce qu'on mesure |
|---|---|---|
| `smolvlm256_zero`, `smolvlm500_zero` | SmolVLM 256M / 500M tel quel | ce que le VLM sait sans exemple |
| `smolvlm256_lora`, `smolvlm500_lora` | affiné par LoRA sur l'entraînement | ce qu'il apprend des 1 568 tuiles |
| `smolvlm500_lora` en 4 bits | même adaptateur, poids quantifiés | la version « compressée » |

Comparaison : réseau de référence `mobilenet_v3_small_512_v3` (72 % · 95 %, 1,4 % de fausses alertes, 6,1 Mo, 15 ms). Durée estimée : environ une heure sur un T4.

Avant de lancer : *Exécution › Modifier le type d'exécution › GPU T4*, et le secret `HF_TOKEN` (écriture) dans la barre latérale.

In [ ]:
HF_DATA = "Elfried9684/onelook-alert-data"
HF_RUNS = "Elfried9684/onelook-alert-runs"
REPO_URL = "https://github.com/elfried96/onelook-alert_Elf.git"
WORKDIR = "/content/onelook-alert_Elf"
MODELS = {  # nom court -> modèle Hugging Face (licence Apache 2.0)
    "smolvlm256": "HuggingFaceTB/SmolVLM-256M-Instruct",
    "smolvlm500": "HuggingFaceTB/SmolVLM-500M-Instruct",
}
EPOCHS = 2

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : changer le type d'exécution"

## 1. Code, environnement et données

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not os.path.exists(WORKDIR):
    !git clone -q {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} pull -q
%cd {WORKDIR}
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --frozen --group vlm
!git log --oneline -1
!uv run hf download {HF_DATA} --repo-type dataset --local-dir data/pack --exclude "pack12/*" --quiet
!uv run python scripts/check_pack.py

Essai éclair (deux pas d'affinage, 16 tuiles de validation) : vérifie la chaîne LoRA sur GPU avant les vrais entraînements. S'il échoue, copier l'erreur complète.

In [ ]:
!uv run --group vlm python scripts/vlm.py train --name essai_vlm --smoke --batch 4
!rm -rf runs/essai_vlm

## 2. Zéro-shot : le VLM tel quel

In [ ]:
for short, model in MODELS.items():
    for split in ("val", "test"):
        !uv run --group vlm python scripts/vlm.py evaluate --name {short}_zero --model {model} --split {split} --batch 16

## 3. Affinage LoRA sur l'entraînement

In [ ]:
for short, model in MODELS.items():
    !uv run --group vlm python scripts/vlm.py train --name {short}_lora --model {model} --epochs {EPOCHS} --batch 8
    for split in ("val", "test"):
        !uv run --group vlm python scripts/vlm.py evaluate --name {short}_lora --model {model} --adapter runs/{short}_lora/adapter --split {split} --batch 16

## 4. Version compressée : poids en 4 bits

In [ ]:
model = MODELS["smolvlm500"]
for split in ("val", "test"):
    !uv run --group vlm python scripts/vlm.py evaluate --name smolvlm500_lora_4bits --model {model} --adapter runs/smolvlm500_lora/adapter --bits 4 --split {split} --batch 16

## 5. Bilan sur le test

In [ ]:
import json, glob, pandas as pd
rows = []
for path in sorted(glob.glob("runs/smolvlm*/eval_test_model.json")):
    r = json.load(open(path))
    rows.append({"run": path.split("/")[1], "rappel inond.": r["rappel_inondation"], "rappel incendie": r["rappel_incendie"],
                 "fausses alertes": r["fausses_alertes"], "fausses alertes eau": r["fausses_alertes_eau"],
                 "précision inond.": r["precision_inondation"], "précision incendie": r["precision_incendie"],
                 "taille Mo": r["taille_mo"]})
pd.DataFrame(rows).set_index("run").round(3)

## 6. Ranger les résultats

Dépôt **privé** Hugging Face, à côté des runs du réseau (adaptateurs LoRA de quelques Mo, décisions, mesures).

In [ ]:
!uv run hf repos create {HF_RUNS} --type model --private --exist-ok
!uv run hf upload {HF_RUNS} runs runs --include "smolvlm*/**" --quiet
print(f"https://huggingface.co/{HF_RUNS}/tree/main/runs")

Ensuite, en local : `uv run hf download Elfried9684/onelook-alert-runs --include "runs/smolvlm*" --local-dir .`, puis le temps sur un cœur de processeur : `scripts/vlm.py evaluate ... --split test --limit 5 --timing 10` dans un dossier d'essai.